# 03 — Membersihkan tabel BPS

Tabel BPS diformat untuk dibaca, bukan untuk diolah. Berkas mentah `bps_penduduk_kecamatan.csv` diunduh
dari halaman tabel statistik BPS (tombol unduh CSV) dan **tidak pernah diedit dengan tangan**: semua
perbaikan ada di skrip, supaya pembersihan kuartal depan sama persis dengan hari ini.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import pandas as pd

from bps import baca_penduduk_bps, kunci_nama
from kota import CFG, OSM, PROC, RAW, UTM

f = RAW / "bps_penduduk_kecamatan.csv"
print(f.read_bytes()[:3], "← tiga bita pertama")
pd.read_csv(f, header=None, encoding="utf-8").head(6)

Masalahnya terlihat: tiga bita tak terlihat di awal (BOM), empat baris judul dan header bertingkat,
nilai sebagai teks, dan baris total di bawah (lihat `tail`). Kota lain dan tahun lain punya variasi sendiri —
misalnya ada atau tidaknya kolom Jumlah, dan "Laki laki" versus "Laki-laki".

In [ ]:
pd.read_csv(f, header=None, encoding="utf-8-sig").tail(3)

## 1. Satu fungsi untuk semua variasi
`scripts/bps.py` mencari baris tahun, menamai kolom dari baris di atasnya, membuang baris total,
mengubah teks menjadi angka, menghitung `jumlah` bila tidak ada — dan **memeriksa** hasilnya terhadap total BPS.

In [ ]:
pdd, cek = baca_penduduk_bps(f)
print(cek)
assert cek.get("total_cocok", True), "jumlah per kecamatan tidak sama dengan total BPS"
pdd

## 2. Menggabungkan dengan batas wilayah OSM
Nama tidak selalu sama antar-sumber. Kunci pencocokan membuang huruf besar, spasi, dan tanda hubung
("Gajah Mungkur" = "Gajahmungkur"). Sisa perbedaan ejaan dicatat di tabel alias — eksplisit, bukan
diperbaiki diam-diam.

In [ ]:
ALIAS = {"panakukkang": "panakkukang"}  # ejaan BPS → ejaan OSM

kec = gpd.read_file(OSM, layer="kecamatan").to_crs(UTM)
kec["luas_km2"] = kec.area / 1e6
kec["kunci"] = kec["kecamatan"].map(kunci_nama)
pdd["kunci"] = pdd["kecamatan"].map(kunci_nama).replace(ALIAS)

gab = kec.merge(pdd.drop(columns="kecamatan"), on="kunci", how="outer", indicator=True)
print(gab["_merge"].value_counts().to_dict())
tanpa_pasangan = gab[gab["_merge"] != "both"]
tanpa_pasangan[["kecamatan", "kunci", "jumlah", "_merge"]]

Setiap baris tanpa pasangan harus dijelaskan sebelum lanjut: ejaan (tambahkan ke `ALIAS`), atau wilayah yang
memang tidak ada di salah satu sumber — misalnya batas kecamatan yang belum dipetakan di OSM. Penduduknya
tidak boleh hilang tanpa catatan.

In [ ]:
if len(tanpa_pasangan):
    hilang = tanpa_pasangan.loc[tanpa_pasangan["_merge"] == "right_only", "jumlah"].sum()
    print(f"penduduk BPS tanpa poligon OSM: {int(hilang):,} jiwa ({hilang / pdd['jumlah'].sum():.1%})")

# Di mana lubangnya? Batas kota dikurangi semua poligon kecamatan = wilayah tanpa kecamatan di OSM.
kota = gpd.read_file(OSM, layer="batas_kota").to_crs(UTM)
sisa = kota.geometry.iloc[0].difference(kec.union_all())
print(f"wilayah kota tanpa poligon kecamatan: {sisa.area / 1e6:.1f} km² "
      f"({sisa.area / kota.area.iloc[0]:.1%} luas kota)")

## 3. Kepadatan penduduk

In [ ]:
ok = gab[gab["_merge"] == "both"].copy()
ok["jiwa_per_km2"] = ok["jumlah"] / ok["luas_km2"]
ok["rasio_jk"] = 100 * ok["laki_laki"] / ok["perempuan"]  # laki-laki per 100 perempuan
print(f"{CFG['name']}, tahun {cek['tahun']}: {int(pdd['jumlah'].sum()):,} jiwa")
ok.sort_values("jiwa_per_km2", ascending=False)[["kecamatan", "jumlah", "luas_km2", "jiwa_per_km2", "rasio_jk"]] \
    .round(1)

In [ ]:
ok[["kecamatan", "kunci", "laki_laki", "perempuan", "jumlah", "tahun", "luas_km2", "jiwa_per_km2"]] \
    .to_csv(PROC / "penduduk_kecamatan.csv", index=False)
print("disimpan: penduduk_kecamatan.csv")